# Прогноз дощу

Мережко Олександр

Навчаю логістичну регресію на попередніх роках і перевіряю на останньому році. Повторний EDA не роблю

In [1]:
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

url = ('https://raw.githubusercontent.com/goitacademy/MACHINE-LEARNING-NEO/'
       'main/datasets/mod_03_topic_05_weather_data.csv.gz')
data = pd.read_csv(url)
print('Розмір даних:', data.shape)
print('Без відомого RainTomorrow:', data['RainTomorrow'].isna().sum())
data = data.dropna(subset=['RainTomorrow']).copy()

Розмір даних: (145460, 23)
Без відомого RainTomorrow: 3267


## Підготовка даних

Прибираю чотири колонки з великою кількістю пропусків, як у конспекті. Рядки без RainTomorrow видалені, бо для них немає правильної відповіді

In [2]:
drop_columns = ['Evaporation', 'Sunshine', 'Cloud9am', 'Cloud3pm']
print('Пропуски у видалених колонках, %:')
print((data[drop_columns].isna().mean() * 100).round(1))
data = data.drop(columns=drop_columns)

data_num = data.select_dtypes(include='number').copy()
data_cat = data.select_dtypes(exclude='number').copy()
data_cat['Date'] = pd.to_datetime(data_cat['Date'])
data_cat['Year'] = data_cat['Date'].dt.year
data_cat['Month'] = data_cat['Date'].dt.month.astype(str)
data_num['Year'] = data_cat.pop('Year')
data_cat = data_cat.drop(columns=['Date', 'RainTomorrow'])
y = data['RainTomorrow']

print('Числові:', data_num.columns.tolist())
print('Категоріальні:', data_cat.columns.tolist())

Пропуски у видалених колонках, %:
Evaporation    42.8
Sunshine       47.7
Cloud9am       37.7
Cloud3pm       40.2
dtype: float64
Числові: ['MinTemp', 'MaxTemp', 'Rainfall', 'WindGustSpeed', 'WindSpeed9am', 'WindSpeed3pm', 'Humidity9am', 'Humidity3pm', 'Pressure9am', 'Pressure3pm', 'Temp9am', 'Temp3pm', 'Year']
Категоріальні: ['Location', 'WindGustDir', 'WindDir9am', 'WindDir3pm', 'RainToday', 'Month']


## Розділення за роком

Рік залишаю числом, місяць — категорією. Останній рік повністю йде в тест

In [3]:
last_year = data_num['Year'].max()
test_mask = data_num['Year'] == last_year
train_mask = ~test_mask

X_train_num = data_num.loc[train_mask]
X_test_num = data_num.loc[test_mask]
X_train_cat = data_cat.loc[train_mask]
X_test_cat = data_cat.loc[test_mask]
y_train = y.loc[train_mask]
y_test = y.loc[test_mask]

print('Навчання:', len(y_train), 'рядків, роки',
      X_train_num['Year'].min(), '—', X_train_num['Year'].max())
print('Тест:', len(y_test), 'рядків, рік', last_year)
print('Частка дощових днів у навчанні:', round(y_train.eq('Yes').mean(), 3))

Навчання: 133727 рядків, роки 2007 — 2016
Тест: 8466 рядків, рік 2017
Частка дощових днів у навчанні: 0.225


## Пропуски та кодування

Числові пропуски замінюю середнім, категоріальні — найчастішим значенням. Усі перетворення навчаю тільки на тренувальних даних

In [4]:
num_imputer = SimpleImputer(strategy='mean')
X_train_num = num_imputer.fit_transform(X_train_num)
X_test_num = num_imputer.transform(X_test_num)

scaler = StandardScaler()
X_train_num = scaler.fit_transform(X_train_num)
X_test_num = scaler.transform(X_test_num)

cat_imputer = SimpleImputer(strategy='most_frequent')
X_train_cat = cat_imputer.fit_transform(X_train_cat)
X_test_cat = cat_imputer.transform(X_test_cat)

encoder = OneHotEncoder(drop='if_binary', handle_unknown='ignore', sparse_output=False)
X_train_cat = encoder.fit_transform(X_train_cat)
X_test_cat = encoder.transform(X_test_cat)

X_train = np.hstack([X_train_num, X_train_cat])
X_test = np.hstack([X_test_num, X_test_cat])
print('Навчальна матриця:', X_train.shape)
print('Тестова матриця:', X_test.shape)
print('Пропусків після обробки:', np.isnan(X_train).sum() + np.isnan(X_test).sum())

Навчальна матриця: (133727, 123)
Тестова матриця: (8466, 123)
Пропусків після обробки: 0


## Модель

Використовую liblinear і balanced, як у прикладі. Дощових днів менше, тому враховую ваги класів

In [5]:
model = LogisticRegression(
    solver='liblinear', class_weight='balanced', random_state=42, max_iter=1000
)
model.fit(X_train, y_train)
pred = model.predict(X_test)
print(classification_report(y_test, pred, digits=2))

              precision    recall  f1-score   support

          No       0.92      0.82      0.87      6703
         Yes       0.51      0.72      0.60      1763

    accuracy                           0.80      8466
   macro avg       0.71      0.77      0.73      8466
weighted avg       0.83      0.80      0.81      8466



## Порівняння

Базові значення взяті з [конспекту](https://www.edu.goit.global/learn/43680690/23598278/24996375/training?blockId=25083067), де дані ділили випадково

In [6]:
report = classification_report(y_test, pred, output_dict=True)
comparison = pd.DataFrame({
    'Конспект': [0.79, 0.51, 0.76, 0.61],
    'Тест за останній рік': [report['accuracy'], report['Yes']['precision'],
                           report['Yes']['recall'], report['Yes']['f1-score']]
}, index=['Accuracy', 'Precision Yes', 'Recall Yes', 'F1 Yes'])
comparison.round(2)

,Конспект,Тест за останній рік
Accuracy,0.79,0.80
Precision Yes,0.51,0.51
Recall Yes,0.76,0.72
F1 Yes,0.61,0.60


## Висновок

Accuracy вийшла 0,80 проти 0,79 у конспекті. Recall для дощу трохи нижчий: 0,72 проти 0,76. Модель знаходить більшість дощових днів, але часто дає хибний прогноз дощу, тому лише accuracy тут недостатньо

Поділ за роком краще відповідає прогнозу на майбутнє. Тестові вибірки різні, тому невелика різниця метрик ще не означає, що модель стала кращою